# Water CAS(8e,6o): state-average DMRG with QNTensorTrains

This notebook reproduces the two block2 state-average DMRG runs in this folder with
[QNTensorTrains](../../QNTensorTrains). Both runs use the active-space integrals in `FCIDUMP`
(water/6-31G, AVAS active space: 6 orbitals, 8 electrons), 3 roots with equal weights 1/3, and
bond dimension `maxM = 50`:

| block2 run | targets | QNTensorTrains layer |
|---|---|---|
| `dmrg_sa_SSS.out` (`spin 0 0 0`) | three singlets | SU(2)-adapted, total spin S = 0 |
| `dmrg_sa_STS.out` (`spin 0 2 0`) | singlet, triplet, singlet | determinant basis, S_z = 0 |

An SU(2) tensor train carries a single total spin, so the spin-mixed STS average is done in the
determinant (S_z = 0) basis, where the three lowest levels are S₀, T₁, S₁.

In [1]:
# QNTensorTrains is expected as a sibling checkout of DMRG_exc_states; set QNTENSORTRAINS_PATH otherwise.
using Pkg
Pkg.activate(get(ENV, "QNTENSORTRAINS_PATH", joinpath(@__DIR__, "..", "..", "QNTensorTrains")))

using LinearAlgebra, Printf, Random
using QNTensorTrains
using QNTensorTrains: SU2, stateAverageMALS!, states

  Activating 

project at `/Volumes/Scratch/Dropbox/Workplace/JuliaPackages/QNTensorTrains`


## Integrals

`read_fcidump` returns `t[i,j] = h_ij`, `v[i,j,k,l] = (ij|kl)` and the core energy `E0` (nuclear
repulsion plus frozen core). The orbitals are put in the order block2 chose (its Fiedler
reordering `[4 5 1 0 2 3]`, printed in the `.out` files), so the tensor train has the same site
order as block2's MPS: site `k` holds orbital `perm[k]`.

In [2]:
fd = read_fcidump(joinpath(@__DIR__, "FCIDUMP"))
E0, d, N = fd.E0, fd.norb, fd.nelec

perm = [5, 6, 2, 1, 3, 4]
t = fd.t[perm, perm]
v = fd.v[perm, perm, perm, perm]

occupied = [perm[k] ≤ N ÷ 2 for k in 1:d]   # reference: orbitals 1-4 doubly occupied
@show d N E0 occupied;

d = 6


N = 8
E0 = -52.416363130011426
occupied = 

Bool[0, 0, 1, 1, 1, 1]


## State-average DMRG

`stateAverageMALS!` runs two-site DMRG sweeps on a `MultiTT`: `nroots` states that share every
core except the orthogonality center. At each bond it solves for the lowest `nroots` eigenpairs of
the local effective Hamiltonian and truncates the shared bond with the weighted density matrix
Σᵢ wᵢ ρᵢ, the same scheme as block2's state-average DMRG.

The settings follow block2's: bond dimension `M = 50`, equal weights, and perturbative
density-matrix noise 1e-4 (block2 sweeps 0–7), then 1e-3 (sweeps 8–15), then 0. One QNTensorTrains
iteration is a forward and a backward sweep, and the loop only stops on a noise-free iteration.

**Start.** The closed-shell reference plus a random perturbation of relative size 1e-2, with rank 1
in every symmetry sector. The random part is needed: the perturbative noise only adds directions
that `H` couples to the current states, so from the bare reference every root stays in the
reference's spatial symmetry, and the lowest excitations (which empty orbital 4) are never found.
Without it, roots 2 and 3 converged to higher eigenstates, 0.07–0.09 Ha too high.

In [3]:
const NOISE = [1e-4, 1e-4, 1e-4, 1e-4, 1e-3, 0.0]

# x0 + ε‖x0‖ p/‖p‖ with p a Gaussian tensor train of rank 1 in every sector
perturbed(x0::TTvector, ε) = perturbation(x0, 1, ε)
function perturbed(x0::SU2.TTvector{T,Nel,twoS,d}, ε) where {T,Nel,twoS,d}
    r = [zeros(Int, Nel + 1, Nel + 1) for _ in 1:d+1]
    for k in 1:d+1, (n, s) in SU2.state_qn(Nel, twoS, d, k)
        r[k][n, s] = 1
    end
    p = SU2.tt_randn(Val(d), Val(Nel), Val(twoS), r)
    return x0 + (ε * norm(x0) / norm(p)) * p
end

function sa_dmrg(H, x0; nroots = 3, M = 50, seed = 1234)
    Random.seed!(seed)
    m = MultiTT(perturbed(x0, 1e-2), nroots; seed)
    λ, m, history = stateAverageMALS!(H, m; weights = fill(1 / nroots, nroots), bondDims = [M],
                                      noise = NOISE, ε = 1e-8, maxIter = 30)
    for (it, h) in enumerate(history)
        @printf("iter %d  noise %.0e  E = %s  max rank %d\n", it, h.noise,
                join([@sprintf("%.10f", e + E0) for e in h.energies], "  "), h.max_rank)
    end
    return λ .+ E0, states(m)
end

sa_dmrg (generic function with 1 method)

## SSS: three singlets (SU(2) layer)

The Hamiltonian is built on the S = 0 sector and the reference is the closed-shell CSF.

In [4]:
H_sss = SU2.SparseHamiltonian(t, v, N, 0, d)
csf = SU2.tt_csf([occ ? 4 : 1 for occ in occupied])   # genealogical code 4 = doubly occupied

E_sss, ψ_sss = sa_dmrg(H_sss, csf);

iter 1  noise 1e-04  E = -75.9942726736  -75.6831483374  -75.5900354207  max rank 50
iter 2  noise 1e-04  E = -75.9942726770  -75.6831483371  -75.5900354223  max rank 50


iter 3  noise 1e-04  E = -75.9942726442  -75.6831483689  -75.6025267683  max rank 50
iter 4  noise 1e-04  E = -75.9942726442  -75.6831483688  -75.6025267683  max rank 50
iter 5  noise 1e-03  E = -75.9942722114  -75.6831478642  -75.6025267683  max rank 50
iter 6  noise 0e+00  E = -75.9942727247  -75.6831483798  -75.6025267683  max rank 22
iter 7  noise 0e+00  E = -75.9942727247  -75.6831483798  -75.6025267683  max rank 22


## STS: singlet, triplet, singlet (S_z = 0 determinant layer)

In [5]:
H_sts = SparseHamiltonian(t, v, N, 0//1, d)
hf = tt_state(occupied, occupied)                     # Hartree-Fock determinant

E_sts, ψ_sts = sa_dmrg(H_sts, hf);

iter 1  noise 1e-04  E = -75.9942724026  -75.7018487979  -75.6831472281  max rank 50
iter 2  noise 1e-04  E = -75.9942724042  -75.7018487968  -75.6831472167  max rank 50
iter 3  noise 1e-04  E = -75.9942724192  -75.7018487865  -75.6831472324  max rank 50
iter 4  noise 1e-04  E = -75.9942724192  -75.7018487865  -75.6831472324  max rank 50
iter 5  noise 1e-03  E = -75.9942440340  -75.7018341267  -75.6831368325  max rank 50
iter 6  noise 0e+00  E = -75.9942727247  -75.7018502933  -75.6831483798  max rank 36
iter 7  noise 0e+00  E = -75.9942727247  -75.7018502933  -75.6831483798  max rank 36
iter 8  noise 0e+00  E = -75.9942727247  -75.7018502933  -75.6831483798  max rank 36


## Comparison with block2

Final energies and the `DMRG OCC` lines (diagonal of the one-particle density matrix, in FCIDUMP
orbital order) from the two `.out` files.

In [6]:
block2 = (sss = (E = [-75.994272293973580, -75.6831488437, -75.6025875550],
                 occ = [[1.998, 1.989, 1.994, 2.000, 0.009, 0.010],
                        [1.994, 1.972, 1.988, 1.000, 1.015, 0.031],
                        [1.994, 1.976, 1.985, 1.000, 0.027, 1.018]]),
          sts = (E = [-75.994272555924653, -75.7018502158, -75.6831275493],
                 occ = [[1.998, 1.989, 1.994, 2.000, 0.009, 0.010],
                        [1.998, 1.973, 1.989, 1.000, 1.010, 0.030],
                        [1.994, 1.972, 1.988, 1.000, 1.015, 0.031]]))

function compare(name, E, ψ, ref)
    @printf("%-6s  %-17s  %-14s  %-8s     %s\n", name, "E(QNTensorTrains)", "E(block2)", "diff",
            "occupations QNTensorTrains / block2")
    for i in eachindex(E)
        occ = zeros(d)
        occ[perm] = real(diag(one_rdm(ψ[i])))           # site k ↦ orbital perm[k]
        @printf("root %d  %-17.10f  %-14.10f  %+.1e  %s / %s\n", i - 1, E[i], ref.E[i], E[i] - ref.E[i],
                join([@sprintf("%.3f", o) for o in occ], " "), join([@sprintf("%.3f", o) for o in ref.occ[i]], " "))
    end
end
compare("SSS", E_sss, ψ_sss, block2.sss)
println()
compare("STS", E_sts, ψ_sts, block2.sts)

SSS     E(QNTensorTrains)  E(block2)       diff         occupations QNTensorTrains / block2


root 0  -75.9942727247     -75.9942722940  -4.3e-07  1.998 1.989 1.994 2.000 0.009 0.010 / 1.998 1.989 1.994 2.000 0.009 0.010


root 1  -75.6831483798     -75.6831488437  +4.6e-07  1.994 1.972 1.988 1.000 1.015 0.031 / 1.994 1.972 1.988 1.000 1.015 0.031
root 2  -75.6025267683     -75.6025875550  +6.1e-05  1.994 1.976 1.985 1.000 0.027 1.018 / 1.994 1.976 1.985 1.000 0.027 1.018

STS     E(QNTensorTrains)  E(block2)       diff         occupations QNTensorTrains / block2


root 0  -75.9942727247     -75.9942725559  -1.7e-07  1.998 1.989 1.994 2.000 0.009 0.010 / 1.998 1.989 1.994 2.000 0.009 0.010
root 1  -75.7018502933     -75.7018502158  -7.7e-08  1.998 1.973 1.989 1.000 1.010 0.030 / 1.998 1.973 1.989 1.000 1.010 0.030
root 2  -75.6831483798     -75.6831275493  -2.1e-05  1.994 1.972 1.988 1.000 1.015 0.031 / 1.994 1.972 1.988 1.000 1.015 0.031


### Full 1-RDMs of the STS run

The STS run also saved its spin-resolved 1-RDMs, to full precision, as `.npy` files in its run
folder `node0/`. That folder was removed from the working tree after commit 416be13, so they are
read from git here. The files are in FCIDUMP orbital order (block2's `orbital_reorder.npy` holds
the same `[4 5 1 0 2 3]` used above), and `E_dmrg.npy` identifies the run.

In [7]:
# Minimal .npy reader (format 1.x, little-endian Float64), so no extra package is needed.
function read_npy(bytes::Vector{UInt8})
    bytes[7] == 0x01 || error("only .npy format version 1.x is handled")
    hlen = Int(bytes[9]) + 256 * Int(bytes[10])
    header = String(bytes[11:10+hlen])
    occursin("'descr': '<f8'", header) && occursin("'fortran_order': False", header) ||
        error("unsupported .npy header: $header")
    shape = Tuple(parse.(Int, split(match(r"'shape': \(([^)]*)\)", header)[1], ",", keepempty = false)))
    data = collect(reinterpret(Float64, bytes[11+hlen:end]))
    isempty(shape) && return data[1]
    return permutedims(reshape(data, reverse(shape)), length(shape):-1:1)   # C order → Julia order
end
node0(file) = read_npy(read(`git -C $(@__DIR__) show 416be13:water_block2/node0/$file`))

@printf("E_dmrg.npy = %.10f  (STS final energy %.10f)\n", node0("E_dmrg.npy"), block2.sts.E[1])

others = [1, 2, 3, 5, 6]
println("        max|γ_block2 - γ_QNTensorTrains|          coupling of orbital 4 to the others")
println("root    all       diagonal  without orbital 4    QNTensorTrains  block2")
for i in 1:3
    γb = dropdims(sum(node0("1pdm-$(i-1)-$(i-1).npy"), dims = 1), dims = 1)   # spin-summed
    γq = zeros(d, d)
    γq[perm, perm] = real(one_rdm(ψ_sts[i]))                                # site order → FCIDUMP order
    Δ = γb - γq
    @printf("%d       %.1e   %.1e   %.1e              %.1e         %.1e\n", i - 1, maximum(abs, Δ),
            maximum(abs, diag(Δ)), maximum(abs, Δ[others, others]),
            maximum(abs, γq[4, others]), maximum(abs, γb[4, others]))
end

E_dmrg.npy = -75.9942725559  (STS final energy -75.9942725559)
        max|γ_block2 - γ_QNTensorTrains|          coupling of orbital 4 to the others
root    all       diagonal  without orbital 4    QNTensorTrains  block2


0       4.1e-05   2.7e-06   3.8e-05              6.3e-15         4.1e-05


1       2.3e-04   2.1e-05   9.0e-05              1.6e-16         2.3e-04
2       1.4e-04   1.7e-05   1.4e-04              1.9e-15         2.6e-05


The diagonals (orbital occupations) agree to 2e-5. Among the other five orbitals, the two sets of
1-RDMs differ by up to 4e-5 (S₀), 9e-5 (T₁) and 1.4e-4 (S₁). The coupling between orbital 4 and the
other five is ≤ 1e-14 in QNTensorTrains' states (and in the dense FCI): with these integrals orbital 4 does
not mix with the others, consistent with it being the out-of-plane lone pair. In block2's states
that coupling reaches 4.1e-5 (S₀) and 2.3e-4 (T₁), the largest disagreement for those two roots.
Either the integrals block2 used mixed orbital 4 with the others, or its states are not converged
at that level. This comparison cannot tell which.

## Check against exact FCI

The active space is small enough for a dense FCI (225 determinants at S_z = 0). It is built here
directly from the integrals by second quantization on bit strings, independently of the
QNTensorTrains Hamiltonian, so it also checks how the FCIDUMP convention is read:

H = E0 + Σ t_ij a†_iσ a_jσ + ½ Σ (ij|kl) a†_iσ a†_kτ a_lτ a_jσ.

A level is a singlet when it does not also appear in the S_z = 1 spectrum (every S ≥ 1 multiplet
has a member there).

In [8]:
# Apply a string of creation (true) / annihilation (false) operators to the determinant b, rightmost
# first. Spin orbital (p, σ) is bit 2(p-1)+σ. Returns (b′, sign), sign = 0 if the result vanishes.
function apply(b, ops)
    s = 1
    for (q, create) in Iterators.reverse(ops)
        isodd(b >> q) == create && return (b, 0)
        isodd(count_ones(b & ((one(b) << q) - 1))) && (s = -s)
        b ⊻= one(b) << q
    end
    return (b, s)
end

function fci_levels(t, v, E0, nup, ndn)
    d = size(t, 1)
    so(p, σ) = 2(p - 1) + σ
    upmask = sum(UInt(1) << so(p, 0) for p in 1:d)
    dets = [b for b in UInt(0):UInt(4^d - 1) if count_ones(b & upmask) == nup && count_ones(b & ~upmask) == ndn]
    index = Dict(b => i for (i, b) in enumerate(dets))
    Hm = Matrix(E0 * I, length(dets), length(dets))
    for (j, b) in enumerate(dets), σ in 0:1, p in 1:d, q in 1:d
        b1, s1 = apply(b, ((so(p, σ), true), (so(q, σ), false)))
        s1 == 0 || (Hm[index[b1], j] += s1 * t[p, q])
        for τ in 0:1, r in 1:d, u in 1:d
            b2, s2 = apply(b, ((so(p, σ), true), (so(r, τ), true), (so(u, τ), false), (so(q, σ), false)))
            s2 == 0 || (Hm[index[b2], j] += s2 * v[p, q, r, u] / 2)
        end
    end
    return eigvals(Symmetric(Hm))
end

E_sz0 = fci_levels(fd.t, fd.v, E0, N ÷ 2, N ÷ 2)
E_sz1 = fci_levels(fd.t, fd.v, E0, N ÷ 2 + 1, N ÷ 2 - 1)
spin(e) = any(abs.(E_sz1 .- e) .< 1e-8) ? "S≥1" : "S=0"

singlets = filter(e -> spin(e) == "S=0", E_sz0)
for (name, E, exact) in (("SSS", E_sss, singlets[1:3]), ("STS", E_sts, E_sz0[1:3]))
    println(name)
    for i in 1:3
        @printf("  root %d  FCI %.10f (%s)   QNTensorTrains - FCI %+.1e   block2 - FCI %+.1e\n", i - 1, exact[i],
                spin(exact[i]), E[i] - exact[i], block2[Symbol(lowercase(name))].E[i] - exact[i])
    end
end

SSS


  root 0  FCI -75.9942727247 (S=0)   QNTensorTrains - FCI -7.1e-14   block2 - FCI +4.3e-07
  root 1  FCI -75.6831483798 (S=0)   QNTensorTrains - FCI -1.1e-13   block2 - FCI -4.6e-07


  root 2  FCI -75.6025267683 (S=0)   QNTensorTrains - FCI -1.6e-13   block2 - FCI -6.1e-05
STS
  root 0  FCI -75.9942727247 (S=0)   QNTensorTrains - FCI -7.1e-14   block2 - FCI +1.7e-07
  root 1  FCI -75.7018502933 (S≥1)   QNTensorTrains - FCI -2.8e-14   block2 - FCI +7.7e-08
  root 2  FCI -75.6831483798 (S=0)   QNTensorTrains - FCI -1.4e-13   block2 - FCI +2.1e-05


**Summary.** QNTensorTrains agrees with FCI to ~1e-13 on all six roots, and its occupations match block2's
to the three printed digits. The bond dimension never binds: the converged ranks are 22 (SU(2))
and 36 (S_z = 0), below `M = 50`. In the SSS history the third root sits on a higher singlet
(−75.5900) for two iterations before the sweep finds the third singlet at −75.6025.

block2's final energies are within 5e-7 of this FCI except the third root of each run, which is
off by 6.1e-5 (SSS) and 2.1e-5 (STS). Some lie *below* it: SSS roots 1 and 2, by 4.6e-7 and
6.1e-5. A state-average DMRG root is a Ritz value and cannot lie below the corresponding exact
level of the Hamiltonian it was computed with, so block2 most likely did not run on exactly these
integrals. Every FCIDUMP committed to this repository (this one, and pyscf's copies in the run
folders deleted after commit 416be13) agrees with it to 6e-16, so the integrals block2 used are
not here. An FCIDUMP written at a different CASSCF orbital iteration would explain the difference.

block2's runs also carry convergence error at the 1e-5 level, independently of the integrals:

- Within the STS run, the printed ground energy rises by 3.0e-5 between the two-site sweeps and
  the final one-site sweeps.
- Seven earlier SSS runs (also deleted after 416be13; maxM 50 to 500) scatter by 1.4e-5 on the
  ground state and 9.5e-5 on the third root.

These files cannot separate how much of each discrepancy comes from the integrals and how much
from convergence.